# Vending-Bench Benchmark — Agentic Decision Making

This notebook runs the **Vending-Bench** benchmark, which tests memory in an agentic setting where the agent manages a simulated vending machine business over 30 days. Better memory should lead to better business decisions (higher P&L).

In [ ]:
# --- Environment setup (must run first) ---
from nb_helpers.config import setup_environment, load_config
setup_environment()

from helpers.types import BenchmarkConfig

# --- Configuration ---
METHODS = ["vdb"]
DAYS = 30
TOP_K = 5
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200
SKIP_INDEX = False

config_dict = load_config(overrides={
    "methods": METHODS,
    "max_questions": DAYS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
})
config = BenchmarkConfig.from_dict(config_dict)

BENCHMARK = "vendingbench"
RUN_CONFIG = {
    "days": DAYS,
    "top_k": TOP_K,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
}
print(f"Config: {len(METHODS)} methods, {DAYS} simulated days, top_k={TOP_K}")

In [ ]:
# --- Load Dataset / Simulator ---
# Placeholder for VendingBench simulator environment setup
from helpers.chunking import chunk_corpus

corpus_text = """
Day 1: Sold 10 Cokes, 5 Pepsis. Weather was hot.
Day 2: Sold 2 Cokes, 8 Pepsis. Weather was rainy.
"""

chunks = chunk_corpus(corpus_text, CHUNK_SIZE, CHUNK_OVERLAP, source="vendingbench_sim")

testset = [
    {
        "question": "Based on past days, how many Cokes should we stock for a hot day?",
        "ground_truth": "10"
    }
]

print(f"VendingBench Environment Loaded: {DAYS} days simulation")

In [ ]:
# --- Run Benchmark Simulation ---
from nb_helpers.pipeline import run_all_methods, save_results

all_results = await run_all_methods(
    methods=METHODS,
    chunks=chunks,
    testset=testset,
    config=config,
    skip_index=SKIP_INDEX,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Evaluate Results ---
from nb_helpers.metrics import evaluate_all_methods

all_results = await evaluate_all_methods(
    all_results,
    config=config,
    use_ragas=False,
    use_em_f1=True,
)
save_results(all_results, BENCHMARK, RUN_CONFIG)

In [ ]:
# --- Summary Table ---
from nb_helpers.charts import summary_table

summary_table(all_results)